In [1]:
# ========================
# FINAL INTEGRATED PROTOTYPE CODE (VIDEO CHAT ONLY)
# ========================
# Run this cell first for installs!
!pip install ipywidgets matplotlib numpy librosa tensorflow scikit-learn transformers sounddevice SpeechRecognition pyttsx3 google-generativeai pydub deepface opencv-python

# --- Step 1: Imports & Setup ---
import ipywidgets as widgets
from IPython.display import display, clear_output, Audio
import matplotlib.pyplot as plt
import numpy as np
import random
import datetime
import librosa
import librosa.display
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, LSTM, Dropout
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from transformers import pipeline
import sounddevice as sd
import speech_recognition as sr
import pyttsx3
import google.generativeai as genai
import threading
import queue
import time
import os
from pydub import AudioSegment
import io
import logging
import cv2 # Added for video capture
from deepface import DeepFace # Added for facial emotion recognition
import sys # For printing status messages to stderr if needed

# For reproducibility and silencing TF/DeepFace warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries loaded.")
# --- Global Config & Resources ---
AUDIO_SAMPLE_RATE = 16000
AUDIO_CHANNELS = 1
AUDIO_CHUNK_DURATION = 0.5
AUDIO_CHUNK_SAMPLES = int(AUDIO_SAMPLE_RATE * AUDIO_CHUNK_DURATION)
STT_PHRASE_TIME_LIMIT = 7
VIDEO_DETECTION_INTERVAL = 5 # Frames between face detection runs
DEEPFACE_DETECTOR_BACKEND = 'opencv' # Fast but less accurate

# Queues & Events
stt_text_q = queue.Queue()
tts_q = queue.Queue()
audio_chunk_q = queue.Queue(maxsize=20)
stop_event = threading.Event()
listen_event = threading.Event()

# Shared State (with locks)
chat_history_lock = threading.Lock()
chat_history = []
voice_emotion_log_lock = threading.Lock()
voice_emotion_log = []
face_emotion_log_lock = threading.Lock()
face_emotion_log = []

# Global emotion holders
last_detected_voice_emotion_global = "N/A"
last_detected_face_emotion_global = "N/A"

# Plotting & Widget variables
ser_fig, ser_ax = None, None
spectrogram_output_widget = None
video_display_widget = None
audio_stream = None
video_capture = None

print("Shared resources initialized.")


D:\anaconda\envs\tf_env\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)



Libraries loaded.
Shared resources initialized.


In [ ]:
# --- (Steps 2, 3, 4: Auth, Intake, Dummy Classification - UNCHANGED) ---
# Keep the exact same code for user_db, create_account, login,
# show_intake_form, train_mental_health_classifier, simulate_classification here...
# --- BEGIN Steps 2,3,4 ---

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

user_db = {} # Simulated user database

def create_account(username, password):
    logging.info(f"Attempting account creation for: {username}")
    if not username or not password: # Add validation
         logging.warning("Signup failed: Username or password empty.")
         return False, "Username and password cannot be empty."
    if username in user_db:
        logging.warning(f"Signup failed: Username '{username}' already exists.")
        return False, "Username already exists."
    else:
        # Simple password "hashing" for prototype - DO NOT USE IN PRODUCTION
        # In production use bcrypt: import bcrypt; hashed = bcrypt.hashpw(password.encode(), bcrypt.gensalt())
        user_db[username] = {"password": password, "data": {}}
        logging.info(f"Account created successfully for: {username}")
        return True, "Account created successfully. Please Login."

def login(username, password):
    logging.info(f"Attempting login for: {username}")
    if not username or not password:
        logging.warning("Login failed: Username or password empty.")
        return False, "Username and password cannot be empty."
    if username in user_db and user_db[username]["password"] == password: # Compare plain text (prototype only)
        # In production: bcrypt.checkpw(password.encode(), user_db[username]["password_hash"].encode())
        logging.info(f"Login successful for: {username}")
        return True, "Login successful."
    else:
        logging.warning(f"Login failed: Invalid credentials for '{username}'.")
        return False, "Invalid username or password."

# Widgets for login (Ensure these are defined only once globally if needed elsewhere)
login_username = widgets.Text(description="Username:")
login_password = widgets.Password(description="Password:")
login_button = widgets.Button(description="Login", button_style='success') # Added style
signup_button = widgets.Button(description="Sign Up", button_style='info') # Added style
login_output = widgets.Output(layout={'border': '1px solid black', 'margin_top': '10px'}) # Added border/margin
main_area = widgets.Output() # Area to display next steps after login
current_username = None # Store logged in user globally

# --- Define Callbacks WITH Error Handling ---
def on_login_clicked(b):
    global current_username
    print("--- Login Clicked ---") # Print to notebook output for basic check
    with login_output: # Ensure output goes to the widget
        clear_output(wait=True) # Clear previous messages in this widget
        logging.info("Login button event triggered.")
        try:
            user = login_username.value
            pwd = login_password.value
            logging.info(f"Attempting login for user: '{user}'")
            success, msg = login(user, pwd)
            print(msg) # Print result message inside the login_output widget
            if success:
                logging.info("Login success branch entered.")
                current_username = user
                # --- UI Transition ---
                login_box.layout.display = 'none' # Hide the login box
                with main_area: # Use the main_area Output widget for the next step
                    clear_output(wait=True) # Clear anything previously in main_area
                    logging.info("Calling show_intake_form...")
                    show_intake_form(user) # Display the intake form HERE
            else:
                 logging.warning("Login failure branch entered.")

        except Exception as e:
             logging.error(f"Error during login process: {e}", exc_info=True) # Log full traceback
             print(f"An internal error occurred during login: {e}") # Show error in widget too

def on_signup_clicked(b):
    print("--- Signup Clicked ---") # Print to notebook output
    with login_output: # Ensure output goes to the widget
        clear_output(wait=True) # Clear previous messages in this widget
        logging.info("Signup button event triggered.")
        try:
            user = login_username.value
            pwd = login_password.value
            logging.info(f"Attempting signup for user: '{user}'")
            success, msg = create_account(user, pwd)
            print(msg) # Print result message inside the login_output widget

        except Exception as e:
            logging.error(f"Error during signup process: {e}", exc_info=True) # Log full traceback
            print(f"An internal error occurred during signup: {e}") # Show error in widget too

# --- Connect Callbacks ---
# Make sure this is done AFTER functions are defined
login_button.on_click(on_login_clicked)
signup_button.on_click(on_signup_clicked)

# --- UI Layout ---
# Ensure login_box and main_area are defined before use in run_prototype
login_box = widgets.VBox([
    widgets.HTML("<h3>Login or Sign Up</h3>"), # Added heading
    login_username,
    login_password,
    widgets.HBox([login_button, signup_button]),
    login_output # Make sure the output widget is part of the displayed box!
])

def show_intake_form(username):
    intake_form_box_widgets = [] # Keep this section the same as before
    age = widgets.IntText(description="Age:", value=30)
    gender = widgets.Dropdown(options=["Male", "Female", "Non-binary", "Other", "Prefer not to say"], description="Gender:")
    ethnicity = widgets.Text(description="Ethnicity (Opt):")
    socioeconomic = widgets.Dropdown(options=["Low", "Medium", "High", "Prefer not to say"], description="Socioeconomic (Opt):")
    sleep = widgets.IntSlider(value=7, min=0, max=14, description="Avg Sleep (hrs):")
    appetite = widgets.Dropdown(options=["Normal", "Decreased", "Increased", "Fluctuates"], description="Appetite Change:")
    energy = widgets.IntSlider(value=5, min=0, max=10, description="Energy Level (0-10):")
    social = widgets.Dropdown(options=["Very Active", "Moderately Active", "Somewhat Withdrawn", "Very Withdrawn"], description="Social Activity:")
    intake_form_box_widgets.extend([age, gender, ethnicity, socioeconomic, sleep, appetite, energy, social])
    submit_button = widgets.Button(description="Submit Intake")
    intake_output = widgets.Output()

    def on_submit_clicked(b):
        with main_area:
             clear_output(wait=True)
             user_db[username]["data"]["intake"] = { "age": age.value, "gender": gender.value, "ethnicity": ethnicity.value, "socioeconomic": socioeconomic.value, "sleep": sleep.value, "appetite": appetite.value, "energy": energy.value, "social": social.value }
             print("Intake data submitted.")
             simulate_classification(username) # Proceeds to classification

    submit_button.on_click(on_submit_clicked)
    intake_box = widgets.VBox([*intake_form_box_widgets, submit_button, intake_output])
    with main_area: clear_output(wait=True); print(f"Welcome {username}! Please provide optional details."); display(intake_box)

def train_mental_health_classifier(): # Keep dummy classifier
    X_train = np.array([[25, 7, 6, 3],[40, 5, 4, 1],[30, 8, 7, 4],[22, 6, 5, 2], [55, 4, 3, 1], [35, 9, 8, 3]])
    y_train = np.array([0, 1, 0, 1, 1, 0]); clf = RandomForestClassifier(n_estimators=50, random_state=42); clf.fit(X_train, y_train); return clf
mental_health_classifier = train_mental_health_classifier()

def simulate_classification(username): # Keep classification logic
    intake = user_db[username]["data"].get("intake", {})
    if not intake: print("No intake data."); start_video_chat_directly(username); return # Go directly to chat if no intake
    features = np.array([[ intake.get("age", 30), intake.get("sleep", 7), intake.get("energy", 5), {"Very Active": 3, "Moderately Active": 2, "Somewhat Withdrawn": 1, "Very Withdrawn": 0}.get(intake.get("social"), 1) ]])
    risk = mental_health_classifier.predict(features)[0]; prob = mental_health_classifier.predict_proba(features)[0][risk]
    issues = {"Depression": 0.0, "Anxiety": 0.0, "Stress & Burnout": 0.0}
    if risk == 1: issues["Depression"] = round(min(1.0, prob + random.uniform(0.1, 0.4)), 2); issues["Anxiety"] = round(min(1.0, prob + random.uniform(0.2, 0.5)), 2); issues["Stress & Burnout"] = round(min(1.0, prob + random.uniform(0.3, 0.6)), 2)
    else: issues["Depression"] = round(max(0.0, prob - random.uniform(0.1, 0.3)), 2); issues["Anxiety"] = round(max(0.0, prob - random.uniform(0.2, 0.4)), 2); issues["Stress & Burnout"] = round(max(0.0, prob - random.uniform(0.1, 0.2)), 2)
    user_db[username]["data"]["issues"] = issues
    print("\nBased on intake data (experimental prediction):"); [print(f" - {issue}: {int(p * 100)}% likelihood") for issue, p in issues.items()]

    # --- CHANGE: Go directly to video chat ---
    start_video_chat_directly(username)
    # --- End CHANGE ---
# --- END Steps 2,3,4 ---


# --- Step 5: Load Models (UNCHANGED) ---
# Keep the loading logic for SER, NLP, LLM, TTS exactly as before...
# --- BEGIN Step 5 ---
FACE_EMOTIONS = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
SER_MODEL_PATH = 'speech_emotion_model.h5'; ser_model = None; ser_encoder = None
SER_EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'ps', 'sad']
try:
    ser_encoder = OneHotEncoder(); ser_encoder.fit(np.array(SER_EMOTIONS).reshape(-1, 1))
    ser_model = load_model(SER_MODEL_PATH)
    print(f"Loaded SER model from {SER_MODEL_PATH}")
except Exception as e: print(f"ERROR loading SER model from {SER_MODEL_PATH}: {e}\nVoice Emotion Disabled."); ser_model = None

def analyze_voice_emotion_keras(audio_chunk): # Keep voice analysis func
    if ser_model is None or ser_encoder is None: return "SER N/A", 0.0
    emotion, confidence = "unknown", 0.0
    try:
        if audio_chunk.dtype != np.float32: audio_chunk = audio_chunk.astype(np.float32) / 32767.0
        if len(audio_chunk) < 2048: return "Too short", 0.0
        mfcc = np.mean(librosa.feature.mfcc(y=audio_chunk, sr=AUDIO_SAMPLE_RATE, n_mfcc=40).T, axis=0)
        mfcc_reshaped = np.expand_dims(np.expand_dims(mfcc, axis=0), axis=-1)
        prediction = ser_model.predict(mfcc_reshaped, verbose=0)
        emotion_index = np.argmax(prediction[0])
        confidence = prediction[0][emotion_index]
        emotion = ser_encoder.inverse_transform(np.array(emotion_index).reshape(1, -1))[0][0]
    except Exception as e: print(f"SER Predict Err: {e}"); emotion = "SER Error"
    return emotion, confidence

print("Loading NLP pipeline..."); nlp_text_analyzer = None # Keep NLP setup
try: nlp_text_analyzer = pipeline("sentiment-analysis"); print("NLP pipeline loaded.")
except Exception as e: print(f"Warn: NLP failed: {e}.")
def analyze_text_nlp_hf(text): # Keep NLP analysis func
     if nlp_text_analyzer:
         try: result = nlp_text_analyzer(text)[0]; return result["label"], result["score"]
         except Exception as e: print(f"NLP Err: {e}"); return "NLP Error", 0.0
     else: return "NEUTRAL", 0.5 # Basic fallback

print("Configuring LLM..."); llm_model = None # Keep LLM setup
try:
    GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')
    if GOOGLE_API_KEY: genai.configure(api_key=GOOGLE_API_KEY); llm_model = genai.GenerativeModel('gemini-pro'); print("Gemini configured.")
    else: print("Warn: GOOGLE_API_KEY missing. LLM disabled.")
except Exception as e: print(f"LLM Err: {e}. LLM disabled.")

print("Initializing TTS engine..."); tts_engine = None # Keep TTS setup
try: tts_engine = pyttsx3.init(); print("TTS initialized.")
except Exception as e: print(f"Warn: TTS init failed: {e}.")
# --- END Step 5 ---

# --- Step 6: Real-time Processing Threads & Functions (UNCHANGED) ---
# Keep audio_callback, audio_processing_thread_ser,
# video_capture_face_emotion_thread, speech_to_text_thread,
# text_to_speech_thread exactly as defined in the previous step...
# --- BEGIN Step 6 ---
def audio_callback(indata, frames, time_info, status): # Keep audio callback
    if status: print("Audio Status:", status, file=sys.stderr)
    if not stop_event.is_set():
        try: audio_chunk_q.put(indata.copy(), timeout=0.05)
        except queue.Full: pass

def audio_processing_thread_ser(): # Keep voice emotion / spectrogram thread
    global ser_fig, ser_ax, last_detected_voice_emotion_global
    print("Audio SER/Spectrogram thread started.")
    while not stop_event.is_set():
        try:
            audio_chunk = audio_chunk_q.get(timeout=0.5)
            if audio_chunk.ndim > 1: audio_chunk = audio_chunk.flatten()
            voice_emotion, ser_confidence = analyze_voice_emotion_keras(audio_chunk)
            last_detected_voice_emotion_global = voice_emotion # Update global
            if voice_emotion not in ["SER N/A", "Too short", "SER Error"]:
                 timestamp = datetime.datetime.now().strftime('%H:%M:%S.%f')[:-3]
                 with voice_emotion_log_lock: voice_emotion_log.append((timestamp, voice_emotion, ser_confidence))
            if ser_fig is not None and ser_ax is not None and spectrogram_output_widget is not None and spectrogram_output_widget.visible:
                 try: # Update spectrogram plot (same plotting code)
                     S = librosa.feature.melspectrogram(y=audio_chunk.astype(np.float32), sr=AUDIO_SAMPLE_RATE, n_mels=128, fmax=8000)
                     S_dB = librosa.power_to_db(S, ref=np.max); ser_ax.clear()
                     librosa.display.specshow(S_dB, sr=AUDIO_SAMPLE_RATE, x_axis='time', y_axis='mel', ax=ser_ax, fmax=8000)
                     ser_ax.set_title(f"Spectrogram (Voice: {voice_emotion})")
                     with spectrogram_output_widget: clear_output(wait=True); display(ser_fig)
                 except Exception as plot_e: pass # Ignore plotting errors occasionally
            audio_chunk_q.task_done()
        except queue.Empty: continue
        except Exception as thread_e: print(f"SER Thread Err: {thread_e}"); time.sleep(0.1)
    print("Audio SER/Spectrogram thread finished.")


def video_capture_face_emotion_thread(): # Keep face emotion / video display thread
    global last_detected_face_emotion_global, video_capture, video_display_widget
    print("Video/Face thread started."); frame_count = 0; last_analysis = {}
    try: video_capture = cv2.VideoCapture(0); assert video_capture.isOpened(), "Webcam Error"
    except Exception as e: print(f"Webcam Init Error: {e}"); stop_event.set(); return
    print("Webcam opened.")
    while not stop_event.is_set():
        ret, frame = video_capture.read()
        if not ret: time.sleep(0.1); continue
        display_frame = frame.copy(); frame_count += 1
        if frame_count % VIDEO_DETECTION_INTERVAL == 0:
            face_emotion = "N/A"; confidence = 0.0; face_region = None
            try:
                results = DeepFace.analyze(img_path=frame, actions=['emotion'], detector_backend=DEEPFACE_DETECTOR_BACKEND, enforce_detection=False, silent=True)
                if results and isinstance(results, list) and results[0]:
                     analysis = results[0]; face_emotion = analysis.get('dominant_emotion', "N/A"); confidence = analysis.get('emotion', {}).get(face_emotion, 0.0); face_region = analysis.get('region')
                     last_analysis = {'emotion': face_emotion, 'confidence': confidence, 'region': face_region}
                     timestamp = datetime.datetime.now().strftime('%H:%M:%S.%f')[:-3] # Log face emotion
                     with face_emotion_log_lock: face_emotion_log.append((timestamp, face_emotion, confidence))
                else: last_analysis = {}; face_emotion = "N/A"
                last_detected_face_emotion_global = face_emotion # Update global
            except Exception as e: last_detected_face_emotion_global = "FER Error"; last_analysis = {}
        if 'region' in last_analysis and last_analysis['region']: # Draw box from last good detection
             r = last_analysis['region']; cv2.rectangle(display_frame, (r['x'], r['y']), (r['x']+r['w'], r['y']+r['h']), (0, 255, 0), 1)
             emo = last_analysis.get('emotion', '?'); conf = last_analysis.get('confidence', 0)
             cv2.putText(display_frame, f"{emo} ({conf*100:.0f}%)", (r['x'], r['y']-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)
        if video_display_widget is not None: # Update video widget
            try: _, encoded_image = cv2.imencode('.jpg', display_frame, [cv2.IMWRITE_JPEG_QUALITY, 80]); video_display_widget.value = encoded_image.tobytes()
            except Exception as e: pass # Ignore occasional widget update error
        time.sleep(0.03)
    print("Video/Face thread finished.")
    if video_capture: video_capture.release(); video_capture = None

stt_recognizer = sr.Recognizer() # Keep STT thread
def speech_to_text_thread():
    print("STT thread started.")
    mic = sr.Microphone(sample_rate=AUDIO_SAMPLE_RATE)
    with mic as source:
        print("STT Adjust Noise...")
        time.sleep(0.5)
        try:
            stt_recognizer.adjust_for_ambient_noise(source, duration=1.0)
            print(f"STT Thresh {stt_recognizer.energy_threshold:.1f}")
        except Exception as e:
            print(f"STT Noise Adjust Fail: {e}")
    while not stop_event.is_set():
        if listen_event.wait(timeout=1.0):
            print("\n>>> STT Listening... <<<"); text = "[STT No Trigger]"
            try:
                with mic as source: audio_data = stt_recognizer.listen(source, phrase_time_limit=STT_PHRASE_TIME_LIMIT, timeout=STT_PHRASE_TIME_LIMIT + 1)
                print(">>> STT Processing... <<<"); text = stt_recognizer.recognize_google(audio_data)
            except sr.WaitTimeoutError: text = "[STT Timeout]"
            except sr.UnknownValueError: text = "[STT Audio Not Understood]"
            except sr.RequestError as e: text = f"[STT API Error]"
            except Exception as e: text = f"[STT Sys Error]"
            finally:
                if text.startswith("["): print(text); # Print errors
                stt_text_q.put(text); listen_event.clear()
    print("STT thread finished.")

def text_to_speech_thread(): # Keep TTS thread
    print("TTS thread started.")
    while not stop_event.is_set():
        try: text = tts_q.get(timeout=0.5)
        except queue.Empty: continue
        try:
            if tts_engine: tts_engine.say(text); tts_engine.runAndWait()
            tts_q.task_done()
        except Exception as e: print(f"TTS Error: {e}"); tts_q.task_done()
    print("TTS thread finished.")
# --- END Step 6 ---


# --- Step 7: Chatbot Logic Function (UNCHANGED) ---
# Keep get_chatbot_reply_final exactly as before, it already handles both emotion types
def get_chatbot_reply_final(user_text, current_voice_emotion, current_facial_emotion):
    global chat_history
    bot_response = "..."
    if user_text.startswith("[STT") and user_text.endswith("]"):
        if "Not Understood" in user_text: return random.choice(["Sorry?", "Didn't get that."])
        elif "Timeout" in user_text: return random.choice(["No sound detected.", "Timed out."])
        else: return "Speech input err."
    text_sentiment, _ = analyze_text_nlp_hf(user_text)
    with chat_history_lock:
        chat_history.append({'role': 'user', 'parts': [user_text]})
        limited_history = chat_history[-10:]
    if llm_model:
        try:
            context_prompt = f"You are an empathetic AI. Be concise. If user mentions severe distress/self-harm, gently advise crisis hotline (988)/professional help. \nContext: Voice Emotion={current_voice_emotion}, Face Emotion={current_facial_emotion}, Text Sentiment={text_sentiment}\nHistory:"
            gemini_content = [{'role': 'user', 'parts': [context_prompt]}, {'role': 'model', 'parts': ["Understood. I'll respond empathetically considering the context."]}]
            gemini_content.extend(limited_history)
            response = llm_model.generate_content(gemini_content, stream=False, safety_settings={'HARASSMENT':'block_none','HATE_SPEECH':'block_none','SEXUALLY_EXPLICIT':'block_none','DANGEROUS_CONTENT':'block_none'})
            if response.parts: bot_response = response.text
            else: print(f"LLM Blocked: {response.prompt_feedback}"); bot_response = "Can we discuss something else?"
        except Exception as e: print(f"LLM Error: {e}"); bot_response = "AI brain error."
    else: # LLM Fallback
        combined = f"V:{current_voice_emotion}, F:{current_facial_emotion}"
        if "hello" in user_text.lower(): bot_response = f"Hi! ({combined})"
        else: bot_response = f"OK. ({combined}). Go on."
    with chat_history_lock:
        chat_history.append({'role': 'model', 'parts': [bot_response]})
        if len(chat_history) > 40: chat_history = chat_history[-10:]
    if "goodbye quit now" in user_text.lower():
        bot_response = "Ending session. Take care!" # Stop initiated later
        if tts_engine: tts_q.put(bot_response); tts_q.join()
        stop_event.set() # Set stop event immediately
    return bot_response


# --- Step 8: Chat Interface Logic ---

# --- REMOVE show_chat_mode_selection function ---
# --- REMOVE text_chat_session function ---
# --- REMOVE voice_chat_session function ---

# --- NEW Function to start video chat directly ---
def start_video_chat_directly(username):
    """ Directly Initializes and displays the video chat UI and starts threads."""
    global chat_history, voice_emotion_log, face_emotion_log, audio_stream, video_capture, stop_event
    global ser_fig, ser_ax, spectrogram_output_widget, video_display_widget # Access global widget refs

    print("Initializing Video Chat Session...")
    with main_area: # Clear previous output (like intake form)
        clear_output(wait=True)
        print("--- Video Chat Session ---")

    # --- UI Setup ---
    status_label_vc = widgets.Label(value="Status: Initializing...")
    video_display_widget = widgets.Image(format='jpeg', width=480)
    video_box = widgets.VBox([widgets.Label("Live Video Feed:"), video_display_widget], layout={'border': '1px solid grey', 'width':'500px'}) # Fixed width?
    spectrogram_output_widget = widgets.Output(layout={'border': '1px solid #ccc', 'width':'400px'}) # Fixed width?
    spectrogram_box = widgets.VBox([widgets.Label("Live Audio Spectrogram:"), spectrogram_output_widget], layout={'margin_top': '0px', 'width':'410px'})
    ser_fig, ser_ax = plt.subplots(figsize=(5, 3)); plt.close(ser_fig) # Setup plot fig
    ser_ax.set_title("Spectrogram - Waiting...")
    with spectrogram_output_widget: display(ser_fig) # Display initial empty plot
    chat_log_output_vc = widgets.Output(layout={'border': '1px solid black', 'height': '250px', 'overflow_y': 'scroll', 'margin_top': '10px'})
    listen_button_vc = widgets.Button(description="🎤 Speak", button_style='info', icon='microphone')
    end_chat_button_vc = widgets.Button(description="End & Report", button_style='danger')
    controls_box = widgets.HBox([listen_button_vc, end_chat_button_vc], layout={'margin_top': '10px'})
    top_row = widgets.HBox([video_box, spectrogram_box]) # Video and spectrogram side-by-side
    video_chat_main_box = widgets.VBox([status_label_vc, top_row, chat_log_output_vc, controls_box])

    # Display the UI in the main area
    with main_area:
        display(video_chat_main_box)

    # --- Start Threads & Streams ---
    status_label_vc.value = "Status: Starting threads..."
    print("Starting threads for video session...")
    stop_event.clear()
    chat_history, voice_emotion_log, face_emotion_log = [], [], [] # Reset logs

    threads = []
    # Order matters less here, but start video/audio sources first maybe
    vid_thread = threading.Thread(target=video_capture_face_emotion_thread); threads.append(vid_thread); vid_thread.start()
    time.sleep(0.5) # Give webcam a moment to initialize before audio maybe?
    stt_thread = threading.Thread(target=speech_to_text_thread); threads.append(stt_thread); stt_thread.start()
    ser_thread = threading.Thread(target=audio_processing_thread_ser); threads.append(ser_thread); ser_thread.start()
    if tts_engine: tts_thread = threading.Thread(target=text_to_speech_thread); threads.append(tts_thread); tts_thread.start()

    try:
        status_label_vc.value = "Status: Starting audio..."
        audio_stream = sd.InputStream(samplerate=AUDIO_SAMPLE_RATE, channels=AUDIO_CHANNELS, blocksize=AUDIO_CHUNK_SAMPLES, callback=audio_callback, dtype='float32')
        audio_stream.start()
        status_label_vc.value = "Status: Ready. Speak or press Speak button."
        print("Audio stream started.")
        with chat_log_output_vc: initial_msg="Bot: Video chat active. How are you today?"; print(initial_msg)
        if tts_engine: tts_q.put(initial_msg)
        chat_history.append({'role': 'model', 'parts': [initial_msg]})
    except Exception as e:
        status_label_vc.value = f"Status: ERROR starting audio: {e}"; print(f"Audio Error: {e}"); stop_event.set(); return

    # --- Button Actions ---
    def on_listen_vc_clicked(b): listen_event.set(); status_label_vc.value = "Status: Listening..."
    def on_end_vchat_clicked(b): status_label_vc.value = "Status: Ending..."; stop_event.set(); listen_button_vc.disabled=True; end_chat_button_vc.disabled=True

    listen_button_vc.on_click(on_listen_vc_clicked)
    end_chat_button_vc.on_click(on_end_vchat_clicked)

    # The actual checking of stt_text_q happens in the run_prototype loop


# --- Step 9: Session Report Generation (UNCHANGED) ---
# Keep generate_report function exactly as before, it includes both logs
def generate_report(username):
    data = user_db[username]["data"]; report_lines = []; report_lines.append("="*40); report_lines.append("  Multi-Modal Chatbot Session Report"); report_lines.append("="*40)
    report_lines.append(f"User: {username}"); report_lines.append(f"Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    report_lines.append("--- Intake Summary ---")
    if data.get("intake"): [report_lines.append(f"  {k.capitalize()}: {v}") for k, v in data["intake"].items()]
    else: report_lines.append("  (No intake data)")
    report_lines.append("\n--- Initial Assessment (Exp) ---")
    if data.get("issues"): [report_lines.append(f"  Potential {k}: {int(v * 100)}% ") for k, v in data["issues"].items()]
    else: report_lines.append("  (No assessment)")
    report_lines.append("\n--- Voice Emotion Log ---") # Voice Log
    if voice_emotion_log:
         report_lines.append("  Timestamp   | Emotion         | Confidence"); report_lines.append("  ------------|-----------------|-----------")
         with voice_emotion_log_lock: log_limit = 30; display_log = voice_emotion_log[-log_limit:]; report_lines.extend([f"  ...(last {log_limit})..."] if len(voice_emotion_log)>log_limit else []); report_lines.extend([f"  {ts} | {emo:<15} | {conf:.2f}" for ts, emo, conf in display_log])
    else: report_lines.append("  (No voice emotions logged)")
    report_lines.append("\n--- Face Emotion Log ---") # Face Log
    if face_emotion_log:
         report_lines.append("  Timestamp   | Emotion         | Confidence"); report_lines.append("  ------------|-----------------|-----------")
         with face_emotion_log_lock: log_limit = 30; display_log = face_emotion_log[-log_limit:]; report_lines.extend([f"  ...(last {log_limit})..."] if len(face_emotion_log)>log_limit else []); report_lines.extend([f"  {ts} | {emo:<15} | {conf:.2f}" for ts, emo, conf in display_log])
    else: report_lines.append("  (No face emotions logged)")
    report_lines.append("\n--- Conversation Transcript ---") # Transcript
    if chat_history: [report_lines.append(f"  {t.get('role','?').capitalize()}: {t.get('parts',[''])[0]}") for t in chat_history]
    else: report_lines.append("  (No conversation)")
    report_lines.append("\n"+"="*40); report_content = "\n".join(report_lines)
    filename = f"{username}_session_report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    try:
        with open(filename, "w", encoding='utf-8') as f:
            f.write(report_content)
        print(f"\nReport generated: '{filename}'")
    except Exception as e:
        print(f"Error generating report: {e}")

    except Exception as e: print(f"\nReport Error: {e}")
    issues = data.get("issues", {}); # Professional Help Suggestion
    if any(prob > 0.65 for prob in issues.values()): print("\nRecommendation: Consider seeking professional help."); print("Resources: 988 Lifeline, Crisis Text Line.")
    else: print("\nRecommendation: Continue self-care.")


# --- Keep end_chat_procedure unchanged (it stops threads/streams) ---
def end_chat_procedure(username):
    print("\nEnding chat session and generating report..."); stop_event.set()
    global audio_stream, video_capture
    if audio_stream is not None: # Stop audio stream
        try: print("Stopping audio..."); audio_stream.stop(); audio_stream.close(); audio_stream = None; print("Audio stopped.")
        except Exception as e: print(f"Audio Stop Err: {e}"); audio_stream = None
    if video_capture is not None: # Stop video capture
        try: print("Releasing video..."); video_capture.release(); video_capture = None; print("Video released.")
        except Exception as e: print(f"Video Release Err: {e}"); video_capture = None
    cv2.destroyAllWindows(); [cv2.waitKey(1) for _ in range(3)] # Close OpenCV windows
    print("Waiting briefly for threads..."); time.sleep(1.5) # Wait for threads
    with main_area: clear_output(wait=True); generate_report(username if username else "default_user"); print("\nSession ended. Run cell again to restart.")
    # Maybe re-display login? login_box.layout.display = 'flex'


# --- Step 10: Main Application Execution Logic (UNCHANGED) ---
# Keep run_prototype exactly as before. It checks the stt_text_q
# and handles the interaction regardless of which chat *type* produced the text.
# Since we now ONLY start the video chat, this loop will manage its interaction.
def run_prototype():
    clear_output(); print("="*50); print(" Multi-Modal Mental Health Chatbot Prototype"); print("(Video Chat Only)"); print("="*50)
    display(login_box)
    display(main_area)
    print("\nPrototype running. Login to start...")
    try:
         print("\nPrototype running. Login to start video chat session.")
         while not stop_event.is_set():
            try:
                 user_text = stt_text_q.get(timeout=0.5)
                 # No need to find chat log widget, print to main_area for simplicity
                 # Or we could pass the chat log widget reference globally from start_video_chat_directly
                 with main_area: # Print interactions below the video/spec UI
                      if user_text and not user_text.startswith("[STT"): # Don't print STT errors directly here
                          print(f"\nYou (voice): {user_text}")

                 if user_text: # Process if text is not empty
                      current_voice = last_detected_voice_emotion_global
                      current_face = last_detected_face_emotion_global
                      bot_reply = get_chatbot_reply_final(user_text, current_voice, current_face)

                      with main_area: print(f"Bot: {bot_reply}")
                      if tts_engine and bot_reply: tts_q.put(bot_reply)

                 stt_text_q.task_done()
                 # Check if the stop event was set by the bot response
                 if stop_event.is_set(): break

            except queue.Empty: time.sleep(0.1); continue # Loop on timeout
            except Exception as main_loop_e: print(f"!! Main Loop Err: {main_loop_e} !!"); time.sleep(1)
    except KeyboardInterrupt: print("\nKeyboard Interrupt! Stopping..."); stop_event.set()
    finally:
         print("Main loop exited.")
         if not stop_event.is_set(): stop_event.set() # Ensure stop is set
         # Explicitly call end procedure if loop exits unexpectedly
         end_chat_procedure(current_username if current_username else "unknown_user")
         print("Prototype Shutdown Complete.")

# --- Start the Prototype ---
run_prototype()

 Multi-Modal Mental Health Chatbot Prototype
(Video Chat Only)


Output()


Prototype running. Login to start...

Prototype running. Login to start video chat session.
